<a href="https://colab.research.google.com/github/jovanfranco/usdmxn-volatilidad/blob/main/clase19_walk_forward.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf

# 1. Ejemplos a mano
print("Pliegues con 1500 días, entrenamiento de 750 y prueba de 150:", (1500 - 750) // 150)
print("Pliegues con 1260 días, entrenamiento de 504 y prueba de 63:", (1260 - 504) // 63)

sharpe_in = {"A": [0.9, 0.3, 0.6], "B": [0.4, 0.7, 0.5]}
ret_out = {"A": [1.2, 0.2, -0.6], "B": [-0.4, -0.8, 0.3]}
elegidas = [max(sharpe_in, key=lambda k: sharpe_in[k][i]) for i in range(3)]
ganadora_in = [sharpe_in[e][i] for i, e in enumerate(elegidas)]
ganadora_out = [ret_out[e][i] for i, e in enumerate(elegidas)]
promedio_out = [(ret_out["A"][i] + ret_out["B"][i]) / 2 for i in range(3)]
print("Elegidas:", elegidas)
print("Sharpe dentro promedio de las elegidas:", round(float(np.mean(ganadora_in)), 3))
print("Rendimiento fuera de las elegidas (%):", ganadora_out, "promedio:", round(float(np.mean(ganadora_out)), 3))
print("Rendimiento fuera promedio de las dos (%):", promedio_out, "promedio:", round(float(np.mean(promedio_out)), 3))

# 2. Datos reales: USD/MXN y la familia de 30 cruces de promedios
precio = yf.download("MXN=X", start="2020-10-07", end="2026-10-22", progress=False)["Close"].squeeze()
rend = precio.pct_change().dropna()
costo = 0.0001   # 1 punto base por cambio de posición

series = {}
for rapido in (5, 10, 20, 30, 40):
    for lento in (50, 75, 100, 150, 200, 250):
        sen = np.sign(precio.rolling(rapido).mean() - precio.rolling(lento).mean())
        sen = sen.where(precio.rolling(lento).mean().notna())
        pos = sen.shift(1).reindex(rend.index)            # la posición de hoy usa información de ayer
        series[f"{rapido}/{lento}"] = (pos * rend - costo * pos.diff().abs())
df = pd.DataFrame(series).dropna()                          # mismo periodo para las 30
comprar = rend.reindex(df.index)                            # comprar y mantener dólares
print("Días disponibles:", len(df), "desde", df.index[0].date(), "hasta", df.index[-1].date())

def sharpe(x):
    return float(x.mean() / x.std() * np.sqrt(252))

# 3. Validación hacia adelante (walk forward)
def walk_forward(df, entren, prueba):
    piezas, elegidas, sh_dentro, mejora_pliegue = [], [], [], []
    for ini in range(0, len(df) - entren - prueba + 1, prueba):
        tr = df.iloc[ini:ini + entren]
        te = df.iloc[ini + entren:ini + entren + prueba]
        sh = tr.mean() / tr.std() * np.sqrt(252)
        mejor = sh.idxmax()                                 # se elige solo con el entrenamiento
        elegidas.append(mejor)
        sh_dentro.append(float(sh.max()))
        piezas.append(pd.DataFrame({"elegida": te[mejor], "promedio_30": te.mean(axis=1)}))
        mejora_pliegue.append(float(te[mejor].mean()))
    return pd.concat(piezas), elegidas, sh_dentro, mejora_pliegue

for entren, prueba in [(504, 63), (756, 126)]:
    oos, elegidas, sh_dentro, por_pliegue = walk_forward(df, entren, prueba)
    anios = len(oos) / 252
    print(f"\n=== Entrenamiento {entren} días, prueba {prueba} días ===")
    print("Pliegues:", len(elegidas), "| Días fuera de muestra:", len(oos), f"({anios:.1f} años)")
    print("Cruces elegidos (cuántas veces):", pd.Series(elegidas).value_counts().to_dict())
    s_el = sharpe(oos["elegida"])
    print("Sharpe fuera de muestra de la elegida:", round(s_el, 2),
          "| error estándar aproximado:", round(1 / np.sqrt(anios), 2),
          "| t aproximada:", round(s_el * np.sqrt(anios), 2))
    print("Sharpe fuera de muestra del promedio de las 30:", round(sharpe(oos["promedio_30"]), 2))
    print("Sharpe de comprar y mantener en el mismo periodo:", round(sharpe(comprar.loc[oos.index]), 2))
    print("Sharpe dentro de muestra promedio de las elegidas:", round(float(np.mean(sh_dentro)), 2))
    print("Eficiencia hacia adelante (fuera entre dentro):", round(s_el / float(np.mean(sh_dentro)), 2))
    print("Pliegues con rendimiento positivo:", int(sum(m > 0 for m in por_pliegue)), "de", len(por_pliegue))

Pliegues con 1500 días, entrenamiento de 750 y prueba de 150: 5
Pliegues con 1260 días, entrenamiento de 504 y prueba de 63: 12
Elegidas: ['A', 'B', 'A']
Sharpe dentro promedio de las elegidas: 0.733
Rendimiento fuera de las elegidas (%): [1.2, -0.8, -0.6] promedio: -0.067
Rendimiento fuera promedio de las dos (%): [0.39999999999999997, -0.30000000000000004, -0.15] promedio: -0.017


/tmp/ipykernel_797/2484638565.py:21: FutureWarning: YF.download() has changed argument auto_adjust default to True
  precio = yf.download("MXN=X", start="2020-10-07", end="2026-10-22", progress=False)["Close"].squeeze()


Días disponibles: 1320 desde 2021-09-23 hasta 2026-10-21

=== Entrenamiento 504 días, prueba 63 días ===
Pliegues: 12 | Días fuera de muestra: 756 (3.0 años)
Cruces elegidos (cuántas veces): {'30/75': 7, '40/75': 3, '40/100': 1, '40/250': 1}
Sharpe fuera de muestra de la elegida: -0.06 | error estándar aproximado: 0.58 | t aproximada: -0.11
Sharpe fuera de muestra del promedio de las 30: 0.19
Sharpe de comprar y mantener en el mismo periodo: 0.17
Sharpe dentro de muestra promedio de las elegidas: 1.15
Eficiencia hacia adelante (fuera entre dentro): -0.05
Pliegues con rendimiento positivo: 7 de 12

=== Entrenamiento 756 días, prueba 126 días ===
Pliegues: 4 | Días fuera de muestra: 504 (2.0 años)
Cruces elegidos (cuántas veces): {'30/75': 3, '40/75': 1}
Sharpe fuera de muestra de la elegida: 0.51 | error estándar aproximado: 0.71 | t aproximada: 0.73
Sharpe fuera de muestra del promedio de las 30: 0.67
Sharpe de comprar y mantener en el mismo periodo: -0.31
Sharpe dentro de muestra prom